# Customer Churn Prediction

## Phase 1: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('E:/Data Analyst/Customer Churn/WA Fn UseC_Telco Customer Churn.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

## Phase 2: First Look at the Data

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## Phase 3: Data Cleaning

In [ ]:
print(df['TotalCharges'].unique()[:20])

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges']=pd.to_numeric(df['TotalCharges'])
print("Null in Column 'TotalCharges' is: ", df['TotalCharges'].isnull().sum())
print("Dtype now:", df['TotalCharges'].dtype)

In [ ]:
median_val=df['TotalCharges'].median()
df['TotalCharges']=df['TotalCharges'].fillna(median_val)
print("Null in Column 'TotalCharges' is(Remaining): ", df['TotalCharges'].isnull().sum())
print("Filled with median:", round(median_val, 2))

In [ ]:
df.drop('customerID', axis=1, inplace=True)
print("Columns remaining are:", df.shape[1])

In [ ]:
print("Duplicates:", df.duplicated().sum())
print("Nulls remaining:", df.isnull().sum().sum())
print("Final shape:", df.shape)

## Phase 4: Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
churn_counts = df['Churn'].value_counts()
print(churn_counts)
print("Churn Rate:",round(churn_counts['Yes'] / len(df) * 100, 2), "%")

In [ ]:
churn_counts.plot(kind="bar",color=['Red','Blue'], edgecolor='black')
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customer")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df.groupby('Contract')['Churn'].value_counts(normalize=True).mul(100).round(1).unstack()

In [ ]:
df.groupby('InternetService')['Churn'].value_counts(normalize=True).mul(100).round(1).unstack()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.boxplot(column='tenure', by='Churn', ax=axes[0])
axes[0].set_title('Tenure vs Churn')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Months')

df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1])
axes[1].set_title('Monthly Charges vs Churn')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Charges')
plt.show()

In [ ]:
senior_churn = df.groupby('SeniorCitizen')['Churn'].value_counts(normalize=True).mul(100).round(1).unstack()
senior_churn

## Phase 5: Visualization

In [ ]:
churned = df[df['Churn'] == 'Yes']['MonthlyCharges']
not_churned = df[df['Churn'] == 'No']['MonthlyCharges']

plt.hist(not_churned, bins=40, alpha=0.6,
         color='#2ecc71', label='Not Churned')
plt.hist(churned, bins=40, alpha=0.6,
         color='#e74c3c', label='Churned')

plt.axvline(churned.mean(), color='red',
            linestyle='--',
            label=f'Churned Avg: ${churned.mean():.0f}')
plt.axvline(not_churned.mean(), color='green',
            linestyle='--',
            label=f'Not Churned Avg: ${not_churned.mean():.0f}')

plt.title('Monthly Charges Distribution by Churn')
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Number of Customers')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
payment_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100).round(1)

payment_churn.sort_values(ascending=False).plot(
    kind='barh', color='#e74c3c', edgecolor='white')

plt.title('Churn Rate by Payment Method')
plt.xlabel('Churn Rate (%)')
plt.tight_layout()
plt.show()

print(payment_churn.sort_values(ascending=False))

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_temp = df[numeric_cols].copy()
df_temp['Churn'] = (df['Churn'] == 'Yes').astype(int)

corr = df_temp.corr().round(2)

sns.heatmap(corr, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0,
            linewidths=0.5)

plt.title('Correlation Between Numeric Variables')
plt.tight_layout()
plt.show()

## Phase 6: Preprocessing

In [ ]:
print("Categorical columns:")
print(df.select_dtypes(include='object').columns.tolist())

print("\nNumeric columns:")
print(df.select_dtypes(include='number').columns.tolist())

In [ ]:
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Churn encoded:")
print(df['Churn'].value_counts())

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded: {col}")

In [ ]:
print("Any object columns remaining:")
print(df.select_dtypes(include='object').columns.tolist())

print("\nDataset after encoding:")
print(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,test_size=0.2,random_state=42,stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Churn rate train:", round(y_train.mean()*100, 1), "%")
print("Churn rate test:", round(y_test.mean()*100, 1), "%")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [ ]:
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete.")
print("\nTraining set after scaling:")
print(X_train[numeric_cols].describe().round(2))

## Phase 7: Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Accuracy: {rf_accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred,target_names=['Not Churned', 'Churned']))

In [ ]:
import xgboost as xgb

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

xgb_model.fit(X_train, y_train)

In [ ]:
xgb_pred = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_pred)

print(f"Accuracy: {xgb_accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred,target_names=['Not Churned', 'Churned']))

In [ ]:
print("=" * 40)
print("MODEL COMPARISON")
print("=" * 40)
print(f"Random Forest : {rf_accuracy*100:.2f}%")
print(f"XGBoost       : {xgb_accuracy*100:.2f}%")

if xgb_accuracy > rf_accuracy:
    print("\nXGBoost performs better")
else:
    print("\nRandom Forest performs better")

## Phase 8: Improving Accuracy

In [ ]:
from sklearn.metrics import roc_auc_score

rf_prob = rf_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, rf_prob)
xgb_auc = roc_auc_score(y_test, xgb_prob)

print(f"Random Forest AUC: {rf_auc:.4f}")
print(f"XGBoost AUC      : {xgb_auc:.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

print("Parameter combinations to test:",
      2 * 3 * 3 * 2)
print("This will take 2-3 minutes. Please wait.")

In [ ]:
xgb_tuned = GridSearchCV(
    xgb.XGBClassifier(random_state=42,
                      eval_metric='logloss',
                      verbosity=0),
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_tuned.fit(X_train, y_train)

print("\nBest parameters found:")
print(xgb_tuned.best_params_)
print(f"\nBest CV AUC: {xgb_tuned.best_score_:.4f}")

In [ ]:
xgb_tuned_pred = xgb_tuned.best_estimator_.predict(X_test)
xgb_tuned_prob = xgb_tuned.best_estimator_.predict_proba(X_test)[:, 1]

tuned_accuracy = accuracy_score(y_test, xgb_tuned_pred)
tuned_auc = roc_auc_score(y_test, xgb_tuned_prob)

print("=" * 45)
print("FINAL MODEL COMPARISON")
print("=" * 45)
print(f"{'Model':<25} {'Accuracy':>10} {'AUC':>8}")
print("-" * 45)
print(f"{'Random Forest':<25} "
      f"{rf_accuracy*100:>9.2f}% {rf_auc:>8.4f}")
print(f"{'XGBoost Default':<25} "
      f"{xgb_accuracy*100:>9.2f}% {xgb_auc:>8.4f}")
print(f"{'XGBoost Tuned':<25} "
      f"{tuned_accuracy*100:>9.2f}% {tuned_auc:>8.4f}")

improvement = (tuned_auc - xgb_auc) * 100
print(f"\nAUC improvement after tuning: "f"+{improvement:.2f}%")

## Phase 9: Visualization of Results

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    (rf_pred, 'Random Forest', 'Blues'),
    (xgb_pred, 'XGBoost Default', 'Oranges'),
    (xgb_tuned_pred, 'XGBoost Tuned', 'Greens')
]

for ax, (pred, name, color) in zip(axes, models):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm,annot=True,fmt='d',cmap=color,ax=ax,
                xticklabels=['Not Churned', 'Churned'],
                yticklabels=['Not Churned', 'Churned'],
                linewidths=0.5,cbar=False)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrix Comparison — All 3 Models',fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150,bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))

models_roc = [
    (rf_prob, f'Random Forest (AUC={rf_auc:.3f})', '#3498db'),
    (xgb_prob, f'XGBoost Default (AUC={xgb_auc:.3f})', '#e74c3c'),
    (xgb_tuned_prob, f'XGBoost Tuned (AUC={tuned_auc:.3f})', '#2ecc71')
]

for prob, label, color in models_roc:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, color=color,linewidth=2, label=label)

plt.plot([0, 1], [0, 1], 'k--',linewidth=1, label='Random Guess (AUC=0.500)')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Model Comparison', fontsize=14,
          fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
importance_df = pd.DataFrame(
    {
        'Feature': X.columns,'Importance': xgb_tuned.best_estimator_.feature_importances_
    }).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'],importance_df['Importance'], color='#3498db', edgecolor='white', alpha=0.8)
plt.xlabel('Importance Score')
plt.title('XGBoost Tuned — Feature Importance',fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150,bbox_inches='tight')
plt.show()

print("\nTop 5 features:")
print(importance_df.head(5).to_string(index=False))

## Section 10: Business Findings Summary

In [ ]:
print("=" * 45)
print("CUSTOMER CHURN PREDICTION — FINDINGS SUMMARY")
print("=" * 45)

print("\n1. CHURN RATE")
churn_rate = df['Churn'].mean() * 100
print(f"   {churn_rate:.1f}% of customers churned.")
print("   RECOMMENDATION: Early intervention program, targeting at-risk customers before they leave.")

print("\n2. CONTRACT TYPE IS THE STRONGEST PREDICTOR")
print("   Month-to-month customers churn at ~42%. One-year contract customers churn at ~11%.")
print("   Two-year contract customers churn at ~3%")
print("   RECOMMENDATION: Incentivize long-term contracts. with discounts or loyalty rewards.")

print("\n3. HIGH MONTHLY CHARGES = HIGH CHURN RISK")
print("   Churned customers pay more per month on average.")
print("   RECOMMENDATION: Review pricing for high-value. customers showing disengagement signals.")

print("\n4. ELECTRONIC CHECK PAYMENT = HIGHEST CHURN")
print("   Electronic check users churn most frequently.")
print("   RECOMMENDATION: Encourage auto-pay enrollment. which correlates with lower churn rates.")

print("\n5. MODEL PERFORMANCE")
print(f"   Best Model: XGBoost[Tuned]")
print(f"   Accuracy: {tuned_accuracy*100:.2f}%")
print(f"   AUC Score: {tuned_auc:.4f}")
print("   This model can identify 80%+ of churners. Before they leave, enabling proactive retention.")
print("\n" + "=" * 93)

In [ ]:
df.to_csv('telco_churn_cleaned.csv', index=False)
print("Clean dataset saved: telco_churn_cleaned.csv")
print(f"Final shape: {df.shape}")